# Introduction to Video Models with Hugging Face 🤗

This notebook focuses on image workflows with the `diffusers` library. We start with image loading and color-channel inspection, then move into diffusion-based image generation with DDPM.

### Key Concepts Covered:
1.  **Image Inspection**: Loading images with PIL, NumPy, OpenCV, and Matplotlib.
2.  **Color Channels**: Exploring RGB components and grayscale conversion.
3.  **Diffusion Models**: Using `DDPMPipeline`, `DDPMScheduler`, and `UNet2DModel`.
4.  **Manual Sampling**: Stepping through a denoising loop to understand reverse diffusion.

---

## Environment Setup
Before starting, ensure you have the necessary environment and dependencies:
- **Environment**: `conda activate hugvenv312`
- **Installation**: `pip install -r requirements.txt`

### Useful Resources
- [Hugging Face Diffusers Documentation](https://huggingface.co/docs/diffusers/index): Official documentation for the `diffusers` library.
- [Hugging Face Model Hub](https://huggingface.co/models): Browse pretrained image and diffusion models.
- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html): Reference for tensors, modules, and GPU execution.

In [19]:
import warnings

# Silence warnings so the notebook output stays focused on the examples.
warnings.filterwarnings("ignore")

print("Jupyter Notebook Initialized")

Jupyter Notebook Initialized


In [20]:
# Capture the runtime details so we know which image-processing paths are available.
import platform
import torch
import torch.nn.functional as F
import subprocess

print(f"--- System Information ---")
print(f"Platform: {platform.platform()}")
print(f"Python:   {platform.python_version()}")
print(f"PyTorch:  {torch.__version__}")

print(f"\n--- GPU/ROCm Accelerators ---")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"Device Count:   {torch.cuda.device_count()}")
    print(f"Primary Device: {torch.cuda.get_device_name(0)}")
else:
    print("Optimization Note: No CUDA-capable GPU detected. Operations will run on CPU.")

--- System Information ---
Platform: Windows-11-10.0.26200-SP0
Python:   3.12.13
PyTorch:  2.9.1+rocm7.2.1

--- GPU/ROCm Accelerators ---
CUDA Available: True
Device Count:   1
Primary Device: AMD Radeon RX 7900 XT


In [21]:
import gradio as gr
import numpy as np
from PIL import Image

In [22]:
gr.__version__

'6.19.0'

### Gradio numbers

In [23]:
def add_numbers(x,y):
    return x + y

def sub_numbers(x,y):
    return x - y

def div_numbers(x,y):
    if y == 0:
        return "Error: Division by zero"
    return x / y

def mul_numbers(x,y):
    return x * y

In [24]:
interface = gr.Interface(
    fn=add_numbers,
    inputs=[gr.Number(label="X"), gr.Number(label="Y")],
    outputs=gr.Number(label="Sum")
)

interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7867

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [25]:
interface_sub = gr.Interface(
    fn=sub_numbers,
    inputs=[gr.Number(label="X"), gr.Number(label="Y")],
    outputs=gr.Number(label="Difference")
)

interface_sub.launch(share=True)

* Running on local URL:  http://127.0.0.1:7868

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [26]:
interface_div = gr.Interface(
    fn=div_numbers,
    inputs=[gr.Number(label="X"), gr.Number(label="Y")],
    outputs=gr.Number(label="Quotient")
)

interface_div.launch(share=True)

* Running on local URL:  http://127.0.0.1:7869

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [27]:
interface_mul = gr.Interface(
    fn=mul_numbers,
    inputs=[gr.Number(label="X"), gr.Number(label="Y")],
    outputs=gr.Number(label="Product")
)

interface_mul.launch(share=True)

* Running on local URL:  http://127.0.0.1:7870

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


### Gradio Text

In [33]:
def reverse_text(text):
    return text[::-1]

print("Python"[::-1])  # Output: nohtyP

nohtyP


In [34]:
interface_txt = gr.Interface(
    fn=reverse_text,
    inputs=gr.Text(label="Input Text"),
    outputs=gr.Text(label="Reversed Text")
)

interface_txt.launch(share=True)

* Running on local URL:  http://127.0.0.1:7872

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


### Gradio Sliders

In [35]:
def slider_example(x):
    return f"Slider value is: {x}"

In [36]:
interface_slider = gr.Interface(
    fn=slider_example,
    inputs=gr.Slider(minimum=0, maximum=100, step=1, label="Slider"),
    outputs=gr.Text(label="Output")
)

interface_slider.launch(share=True)

* Running on local URL:  http://127.0.0.1:7873

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


### Gradio Dropdown menu

In [37]:
def dropdown_example(choice):
    return f"You selected: {choice}"

In [38]:
dropdown_interface = gr.Interface(
    fn=dropdown_example,
    inputs=gr.Dropdown(choices=["Option 1", "Option 2", "Option 3"], label="Select an Option"),
    outputs=gr.Text(label="Output")
)

dropdown_interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7874

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
